# Sales Data Cleaning (FIP Deliverable B5)

This notebook takes the messy sales file in `data/raw/sales_raw.csv` and turns it into a clean file
that is ready for analysis (`data/cleaned/sales_cleaned.csv`).

Rules I followed:
- The raw file is **only read, never changed**. All cleaning happens on a copy in memory.
- Every step says **what** was done, **why**, and **how many rows** it affected.
- Running the notebook from top to bottom (Kernel → Restart & Run All) always gives the same result.

## 1. Setup

In [1]:
import hashlib
import pandas as pd

RAW_FILE = "../data/raw/sales_raw.csv"
CLEAN_FILE = "../data/cleaned/sales_cleaned.csv"
PROFILE_FILE = "../docs/data_profile.md"


def file_hash(path):
    """Fingerprint of a file. If the file changes even a little, the hash changes."""
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


# save the fingerprint now so we can prove at the end that the raw file was not touched
raw_hash_start = file_hash(RAW_FILE)
print("Raw file hash at start:", raw_hash_start)

Raw file hash at start: 3aed2ae03719f83755896a5a79ccbd339f9580ac15c812ee14c7f79509256a56


## 2. Load the raw data

Everything is loaded as text (`dtype=str`) so pandas doesn't guess types and quietly change values.
I'll convert each column myself later. `raw` is kept as it is for the before/after comparison, and all the work is done on `df`.

In [2]:
raw = pd.read_csv(RAW_FILE, dtype=str)
df = raw.copy()

print("Rows:", raw.shape[0], "| Columns:", raw.shape[1])
raw.head(10)

Rows: 120 | Columns: 8


,Order ID,Order Date,customer name,City,Product,Qty,Unit Price,Payment Method
0,ORD1091,2024-01-10,NaN,Ibadan,Rice 50kg,4,N78000,Cash
1,ORD1027,2024-06-04,Aisha Lawal,NaN,Garri 10kg,3,"₦12,000",cash
2,ORD1103,2024-04-12,Zainab Okafor,port harcourt,garri 10kg,7,NaN,transfer
3,ORD1015,03/04/2024,Blessing Okafor,Lagos,rice 50kg,10,78000,pos
4,ORD1051,2024-04-07,Ngozi Lawal,Abj,Beans 5kg,10,"₦11,500",POS
5,ORD1047,12/01/2024,Kemi Balogun,NaN,Veg Oil 5L,7,"14,500.00",Bank Transfer
6,ORD1095,10/04/2024,Zainab Eze,lagos,Indomie Carton,7,N9800,Transfer
7,ORD1076,January 21 2024,Halima Eze,NaN,Vegetable Oil 5L,3,"₦14,500",Bank Transfer
8,ORD1098,January 12 2024,Halima Adeyemi,P/H,RICE 50KG,1,"₦78,000",Transfer
9,ORD1055,2024-05-14,MUSA BALOGUN,Abj,Vegetable Oil 5L,10,"14,500.00",Bank Transfer


## 3. Profile the raw data (before)

Looking at the data first so I know what needs fixing.

In [3]:
def profile(data):
    """One row per column: type, missing values and number of different values."""
    return pd.DataFrame({
        "type": data.dtypes.astype(str),
        "missing": data.isna().sum(),
        "missing_%": (data.isna().mean() * 100).round(1),
        "unique_values": data.nunique(),
    })


def summary(data, id_col):
    return {
        "rows": len(data),
        "columns": data.shape[1],
        "missing cells": int(data.isna().sum().sum()),
        "exact duplicate rows": int(data.duplicated().sum()),
        "duplicate order IDs": int(data[id_col].duplicated().sum()),
    }


before_profile = profile(raw)
before_summary = summary(raw, "Order ID")
print(before_summary)
before_profile

{'rows': 120, 'columns': 8, 'missing cells': 29, 'exact duplicate rows': 6, 'duplicate order IDs': 8}


,type,missing,missing_%,unique_values
Order ID,str,0,0.0,112
Order Date,str,2,1.7,102
customer name,str,5,4.2,100
City,str,9,7.5,18
Product,str,0,0.0,19
Qty,str,1,0.8,14
Unit Price,str,8,6.7,23
Payment Method,str,4,3.3,10


In [4]:
# a quick look at how messy some of the text columns are
for col in ["City", "Product ", "Payment Method"]:
    print(col.strip(), "->", sorted(raw[col].dropna().unique()))
    print()

City -> [' Lagos ', 'ABUJA', 'Abj', 'Abuja', 'Ibadan', 'KANO', 'Kano', 'LAGOS', 'Lagos', 'Lagos ', 'P/H', 'PH', 'Port Harcourt', 'abuja', 'ibadan', 'kano', 'lagos', 'port harcourt']

Product -> ['BEANS 5KG', 'Beans 5kg', 'Gari 10kg', 'Garri 10kg', 'Indomie Carton', 'Indomie ctn', 'RICE 50KG', 'Rice 50 kg', 'Rice 50kg', 'Sugar 1Kg', 'Sugar 1kg', 'Veg Oil 5L', 'Vegetable Oil 5L', 'beans 5kg', 'garri 10kg', 'indomie carton', 'rice 50kg', 'sugar 1kg', 'vegetable oil 5l']

Payment Method -> ['Bank Transfer', 'CASH', 'Cash', 'POS', 'Pos ', 'Transfer', 'cash', 'pos', 'transfer', 'trf']



**What I noticed:**
- Column names have extra spaces and mixed case (`" Order Date"`, `"Product "`).
- City, product and payment method are spelt in many ways (`lagos`, `LAGOS`, `Abj`, `PH`, `trf`, ...).
- Dates come in 3 formats: `2024-03-05`, `05/03/2024` and `March 5 2024`. One date doesn't exist (`2024-02-30`).
- Prices are text with `₦`, `N` and commas.
- Quantity has negative numbers, a blank and the word `two`.
- There are missing values and duplicate rows.

## 4. Cleaning steps

Each step adds a line to a log so there's a record of what changed. The full log is shown at the end
and also written up in `docs/decision_log.md`.

In [5]:
log = []

def note(step, rows_affected, reason):
    log.append({"step": step, "rows_affected": rows_affected, "reason": reason})
    print(f"{step}: {rows_affected} rows affected")

### Step 1: Fix column names
Strip spaces, make lowercase, and use `_` instead of spaces, so columns are easy to type in code (`df.order_date`).
`qty` is renamed to `quantity` to make it clearer.

In [6]:
old_names = list(df.columns)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.rename(columns={"qty": "quantity"})

print(dict(zip(old_names, df.columns)))
note("Rename columns", 0, "Only column names changed, no data. Names had extra spaces and mixed case")

{'Order ID': 'order_id', ' Order Date': 'order_date', 'customer name': 'customer_name', 'City': 'city', 'Product ': 'product', 'Qty': 'quantity', 'Unit Price': 'unit_price', 'Payment Method': 'payment_method'}
Rename columns: 0 rows affected


### Step 2: Remove extra spaces from all text
Values like `" Lagos "` and `"  MUSA BALOGUN  "` would count as different from `"Lagos"` and `"Musa Balogun"`.
I remove spaces at the start and end and turn double spaces into single ones. Cells that become empty are treated as missing.

In [7]:
before = df.copy()
for col in df.columns:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
df = df.mask(df == "")

changed = (before.fillna("") != df.fillna("")).any(axis=1).sum()
note("Trim spaces", int(changed), "Extra spaces made the same value look different")

Trim spaces: 28 rows affected


### Step 3: Customer names
Put names in Title Case (`aisha bello` → `Aisha Bello`).
Missing names are filled with `"Unknown"`. I didn't delete these rows because the sale still happened and the money still counts.

In [8]:
missing_names = df["customer_name"].isna().sum()
changed = (df["customer_name"].notna() & (df["customer_name"] != df["customer_name"].str.title())).sum()

df["customer_name"] = df["customer_name"].str.title().fillna("Unknown")

note("Customer name to Title Case", int(changed), "Same person was written in different cases")
note("Fill missing customer name", int(missing_names), "Sale is still valid without a name, so keep the row")

Customer name to Title Case: 38 rows affected
Fill missing customer name: 5 rows affected


### Step 4: City
Title Case first, then fix short forms: `Abj` → `Abuja`, `PH` / `P/H` → `Port Harcourt`.
Missing cities are filled with `"Unknown"` because there is no safe way to guess where a customer is.

In [9]:
city_fix = {"Abj": "Abuja", "Ph": "Port Harcourt", "P/H": "Port Harcourt"}

before = df["city"].copy()
missing_city = df["city"].isna().sum()

df["city"] = df["city"].str.title().replace(city_fix).fillna("Unknown")

changed = ((before != df["city"]) & before.notna()).sum()
note("Standardise city names", int(changed), "Same city spelt in different ways")
note("Fill missing city", int(missing_city), "Can't guess the city, so mark it Unknown")
print(df["city"].value_counts())

Standardise city names: 57 rows affected
Fill missing city: 9 rows affected
city
Port Harcourt    24
Ibadan           23
Abuja            22
Kano             22
Lagos            20
Unknown           9
Name: count, dtype: int64


### Step 5: Product
Map every spelling to one official product name. I lowercase the value first, then look it up in a table.
If a product isn't in the table, the check below stops the notebook so it can't slip through.

In [10]:
product_fix = {
    "rice 50kg": "Rice 50kg", "rice 50 kg": "Rice 50kg",
    "vegetable oil 5l": "Vegetable Oil 5L", "veg oil 5l": "Vegetable Oil 5L",
    "sugar 1kg": "Sugar 1kg",
    "indomie carton": "Indomie Carton", "indomie ctn": "Indomie Carton",
    "garri 10kg": "Garri 10kg", "gari 10kg": "Garri 10kg",
    "beans 5kg": "Beans 5kg",
}

before = df["product"].copy()
df["product"] = df["product"].str.lower().map(product_fix)

assert df["product"].notna().all(), "Found a product that is not in product_fix"
note("Standardise product names", int((before != df["product"]).sum()), "Same product spelt in different ways")
print(df["product"].value_counts())

Standardise product names: 79 rows affected


product
Indomie Carton      28
Rice 50kg           22
Beans 5kg           18
Vegetable Oil 5L    18
Garri 10kg          17
Sugar 1kg           17
Name: count, dtype: int64


### Step 6: Payment method
Group into 3 options: `Cash`, `POS`, `Bank Transfer`. (`transfer`, `bank transfer` and `trf` all mean a bank transfer.)
Missing values become `"Unknown"`.

In [11]:
payment_fix = {"cash": "Cash", "pos": "POS", "transfer": "Bank Transfer",
               "bank transfer": "Bank Transfer", "trf": "Bank Transfer"}

before = df["payment_method"].copy()
missing_pay = df["payment_method"].isna().sum()

df["payment_method"] = df["payment_method"].str.lower().map(payment_fix).fillna("Unknown")

changed = ((before != df["payment_method"]) & before.notna()).sum()
note("Standardise payment method", int(changed), "Same payment type written in different ways")
note("Fill missing payment method", int(missing_pay), "Payment type not recorded")
print(df["payment_method"].value_counts())

Standardise payment method: 82 rows affected
Fill missing payment method: 4 rows affected
payment_method
Bank Transfer    55
Cash             36
POS              25
Unknown           4
Name: count, dtype: int64


### Step 7: Order date
Dates come in three formats. I try each format in turn, which is safer than letting pandas guess.
`05/03/2024` is read as **day/month/year** (5 March) because that's how dates are written in Nigeria.
Anything that doesn't match (blank, or `2024-02-30`, which doesn't exist) becomes `NaT` (missing) and is dealt with in Step 11.

In [12]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%B %d %Y"]

def parse_date(value):
    if pd.isna(value):
        return pd.NaT
    for fmt in DATE_FORMATS:
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            pass
    return pd.NaT

original_dates = df["order_date"].copy()
df["order_date"] = pd.to_datetime(df["order_date"].apply(parse_date))

bad_dates = df["order_date"].isna()
note("Convert order_date to a date", int(len(df) - bad_dates.sum()), "Three different date formats in one column")
print("Dates that could not be read:")
print(original_dates[bad_dates].fillna("(blank)").to_string())

Convert order_date to a date: 117 rows affected


Dates that could not be read:
29       (blank)
50    2024-02-30
84       (blank)


### Step 8: Unit price
Remove `₦`, `N` and commas, then turn the value into a number.

For missing prices: every product has one fixed price (for example Rice 50kg is always 78,000),
so I fill a missing price with the **usual (median) price of that product**. The table below shows that each product has only one price, so this is safe.

In [13]:
missing_price = df["unit_price"].isna().sum()

df["unit_price"] = (df["unit_price"]
                    .str.replace("₦", "", regex=False)
                    .str.replace("N", "", regex=False)
                    .str.replace(",", "", regex=False))
df["unit_price"] = pd.to_numeric(df["unit_price"])

# proof that each product only has one price
print(df.groupby("product")["unit_price"].agg(["min", "max", "count"]))

df["unit_price"] = df["unit_price"].fillna(df.groupby("product")["unit_price"].transform("median"))

note("Convert unit_price to a number", len(df), "Prices were text with symbols and commas")
note("Fill missing unit_price", int(missing_price), "Each product has one fixed price, so use it")

                      min      max  count
product                                  
Beans 5kg         11500.0  11500.0     17
Garri 10kg        12000.0  12000.0     16
Indomie Carton     9800.0   9800.0     26
Rice 50kg         78000.0  78000.0     20
Sugar 1kg          1800.0   1800.0     17
Vegetable Oil 5L  14500.0  14500.0     16
Convert unit_price to a number: 120 rows affected
Fill missing unit_price: 8 rows affected


### Step 9: Quantity
Convert to a number. Text like `two` can't be converted, so it becomes missing (`errors="coerce"`).
Missing and negative quantities are handled in Step 11.

In [14]:
original_qty = df["quantity"].copy()
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

bad_qty = df["quantity"].isna() | (df["quantity"] <= 0)
print("Quantities that are missing, not a number or not above 0:")
print(original_qty[bad_qty].fillna("(blank)").to_string())
note("Convert quantity to a number", len(df), "Quantity was stored as text")

Quantities that are missing, not a number or not above 0:
31          -3
35     (blank)
60          -7
104         -8
115        two
Convert quantity to a number: 120 rows affected


### Step 10: Remove duplicates
This is done **after** the text clean-up on purpose. Before Step 2 and Step 3, `"  MUSA BALOGUN  "` and `"Musa Balogun"`
looked different, so some copies of the same order wouldn't have been caught.

1. Remove rows that are exactly the same.
2. Check that every `order_id` now appears only once. An order ID should be unique, so if one appears twice I keep the first.

In [15]:
dupes = df[df.duplicated(keep=False)].sort_values("order_id")
print("Duplicate rows found:")
display(dupes[["order_id", "customer_name", "product", "quantity", "unit_price"]])

rows_before = len(df)
df = df.drop_duplicates()
note("Remove exact duplicate rows", rows_before - len(df), "Same order entered more than once would double count sales")

rows_before = len(df)
df = df.drop_duplicates(subset="order_id", keep="first")
note("Remove repeated order IDs", rows_before - len(df), "Order ID must be unique")

Duplicate rows found:


,order_id,customer_name,product,quantity,unit_price
89,ORD1002,Ngozi Ibrahim,Indomie Carton,10.0,9800.0
97,ORD1002,Ngozi Ibrahim,Indomie Carton,10.0,9800.0
88,ORD1008,Zainab Ibrahim,Indomie Carton,5.0,9800.0
107,ORD1008,Zainab Ibrahim,Indomie Carton,5.0,9800.0
64,ORD1034,Yusuf Ibrahim,Garri 10kg,5.0,12000.0
90,ORD1034,Yusuf Ibrahim,Garri 10kg,5.0,12000.0
9,ORD1055,Musa Balogun,Vegetable Oil 5L,10.0,14500.0
55,ORD1055,Musa Balogun,Vegetable Oil 5L,10.0,14500.0
67,ORD1082,Segun Ibrahim,Rice 50kg,5.0,78000.0
87,ORD1082,Segun Ibrahim,Rice 50kg,5.0,78000.0


Remove exact duplicate rows: 8 rows affected
Remove repeated order IDs: 0 rows affected


### Step 11: Remove rows that can't be trusted
- **No valid date**: these rows can't be used for sales-over-time analysis, and I can't guess the real date.
- **Quantity missing, not a number, or zero/negative**: I can't tell whether `-3` means 3 items, a refund, or a typo,
  so I don't guess. The rows are removed and listed here so they can be checked with the shop.

This only removes a few rows, and they're printed below so nothing disappears silently.

In [16]:
bad_rows = df["order_date"].isna() | df["quantity"].isna() | (df["quantity"] <= 0)
print("Rows removed:")
display(df[bad_rows][["order_id", "order_date", "product", "quantity"]])

note("Remove rows with no valid date", int(df["order_date"].isna().sum()), "Date missing or impossible")
note("Remove rows with invalid quantity", int((df["quantity"].isna() | (df["quantity"] <= 0)).sum()),
     "Quantity missing, not a number or not above 0")

df = df[~bad_rows].copy()
df["quantity"] = df["quantity"].astype(int)

Rows removed:


,order_id,order_date,product,quantity
29,ORD1112,NaT,Rice 50kg,5.0
31,ORD1035,2024-06-22,Sugar 1kg,-3.0
35,ORD1016,2024-05-04,Sugar 1kg,NaN
50,ORD1041,NaT,Beans 5kg,4.0
60,ORD1101,2024-05-20,Beans 5kg,-7.0
84,ORD1086,NaT,Indomie Carton,9.0
104,ORD1052,2024-06-26,Rice 50kg,-8.0
115,ORD1078,2024-02-01,Sugar 1kg,NaN


Remove rows with no valid date: 3 rows affected
Remove rows with invalid quantity: 5 rows affected


### Step 12: Add total amount and tidy up
- New column `total_amount = quantity × unit_price`, which is what most analysis will need.
- Put the columns in a sensible order and sort by date.

In [17]:
df["total_amount"] = df["quantity"] * df["unit_price"]

df = df[["order_id", "order_date", "customer_name", "city", "product",
         "quantity", "unit_price", "total_amount", "payment_method"]]
df = df.sort_values(["order_date", "order_id"]).reset_index(drop=True)

note("Add total_amount column", len(df), "Needed for revenue analysis")
df.head(10)

Add total_amount column: 104 rows affected


,order_id,order_date,customer_name,city,product,quantity,unit_price,total_amount,payment_method
0,ORD1045,2024-01-02,Ibrahim Adeyemi,Ibadan,Sugar 1kg,7,1800.0,12600.0,Bank Transfer
1,ORD1048,2024-01-05,Uche Usman,Ibadan,Vegetable Oil 5L,1,14500.0,14500.0,POS
2,ORD1054,2024-01-05,Tunde Ibrahim,Abuja,Garri 10kg,8,12000.0,96000.0,Bank Transfer
3,ORD1080,2024-01-09,Ngozi Okonkwo,Port Harcourt,Rice 50kg,4,78000.0,312000.0,Bank Transfer
4,ORD1091,2024-01-10,Unknown,Ibadan,Rice 50kg,4,78000.0,312000.0,Cash
5,ORD1043,2024-01-12,Blessing Adeyemi,Kano,Garri 10kg,8,12000.0,96000.0,POS
6,ORD1047,2024-01-12,Kemi Balogun,Unknown,Vegetable Oil 5L,7,14500.0,101500.0,Bank Transfer
7,ORD1098,2024-01-12,Halima Adeyemi,Port Harcourt,Rice 50kg,1,78000.0,78000.0,Bank Transfer
8,ORD1075,2024-01-14,Blessing Okafor,Unknown,Indomie Carton,7,9800.0,68600.0,Cash
9,ORD1036,2024-01-15,Kemi Eze,Port Harcourt,Garri 10kg,2,12000.0,24000.0,Cash


## 5. Save the cleaned data

In [18]:
df.to_csv(CLEAN_FILE, index=False, date_format="%Y-%m-%d")
print("Saved", len(df), "rows to", CLEAN_FILE)

Saved 104 rows to ../data/cleaned/sales_cleaned.csv


## 6. Check the cleaned file

The cleaned file is loaded back from disk and tested. If any check fails, the notebook stops with an error.

In [19]:
clean = pd.read_csv(CLEAN_FILE, parse_dates=["order_date"])

assert clean.isna().sum().sum() == 0, "There are still missing values"
assert clean.duplicated().sum() == 0, "There are still duplicate rows"
assert clean["order_id"].is_unique, "Order IDs are not unique"
assert (clean["quantity"] > 0).all(), "Found quantity of 0 or less"
assert (clean["unit_price"] > 0).all(), "Found price of 0 or less"
assert set(clean["payment_method"]) <= {"Cash", "POS", "Bank Transfer", "Unknown"}
assert (clean["total_amount"] == clean["quantity"] * clean["unit_price"]).all()

print("All checks passed.")

All checks passed.


## 7. Profile the cleaned data (after) and compare

In [20]:
after_profile = profile(clean)
after_summary = summary(clean, "order_id")

comparison = pd.DataFrame({"before": before_summary, "after": after_summary})
comparison["change"] = comparison["after"] - comparison["before"]
comparison

,before,after,change
rows,120,104,-16
columns,8,9,1
missing cells,29,0,-29
exact duplicate rows,6,0,-6
duplicate order IDs,8,0,-8


In [21]:
after_profile

,type,missing,missing_%,unique_values
order_id,str,0,0.0,104
order_date,datetime64[us],0,0.0,78
customer_name,str,0,0.0,79
city,str,0,0.0,6
product,str,0,0.0,6
quantity,int64,0,0.0,10
unit_price,float64,0,0.0,6
total_amount,float64,0,0.0,52
payment_method,str,0,0.0,4


In [22]:
# the text columns now only have clean values
for col in ["city", "product", "payment_method"]:
    print(col, "->", sorted(clean[col].unique()))

city -> ['Abuja', 'Ibadan', 'Kano', 'Lagos', 'Port Harcourt', 'Unknown']
product -> ['Beans 5kg', 'Garri 10kg', 'Indomie Carton', 'Rice 50kg', 'Sugar 1kg', 'Vegetable Oil 5L']
payment_method -> ['Bank Transfer', 'Cash', 'POS', 'Unknown']


In [23]:
clean[["quantity", "unit_price", "total_amount"]].describe().round(2)

,quantity,unit_price,total_amount
count,104.00,104.00,104.00
mean,5.53,22468.27,119615.38
std,2.87,26630.97,165897.11
min,1.00,1800.00,1800.00
25%,3.00,9800.00,24000.00
50%,6.00,11500.00,69000.00
75%,8.00,14500.00,104625.00
max,10.00,78000.00,780000.00


## 8. Log of every change

In [24]:
log_table = pd.DataFrame(log)
log_table

,step,rows_affected,reason
0,Rename columns,0,"Only column names changed, no data. Names had ..."
1,Trim spaces,28,Extra spaces made the same value look different
2,Customer name to Title Case,38,Same person was written in different cases
3,Fill missing customer name,5,"Sale is still valid without a name, so keep th..."
4,Standardise city names,57,Same city spelt in different ways
5,Fill missing city,9,"Can't guess the city, so mark it Unknown"
6,Standardise product names,79,Same product spelt in different ways
7,Standardise payment method,82,Same payment type written in different ways
8,Fill missing payment method,4,Payment type not recorded
9,Convert order_date to a date,117,Three different date formats in one column


## 9. Write the profile report

Saves the before/after tables to `docs/data_profile.md`, so the report always matches the latest run.

In [25]:
def to_md(table):
    """Turn a DataFrame into a markdown table (written by hand so no extra package is needed)."""
    table = table.reset_index()
    lines = ["| " + " | ".join(str(c) for c in table.columns) + " |",
             "|" + "---|" * len(table.columns)]
    for _, row in table.iterrows():
        lines.append("| " + " | ".join(str(v) for v in row) + " |")
    return "\n".join(lines)

report = f"""# Data Profile: Before and After Cleaning

This file is created automatically by `notebook/data_cleaning.ipynb` (section 9). Don't edit it by hand.

## Summary

{to_md(comparison.rename_axis("measure"))}

## Raw data (before)

File: `data/raw/sales_raw.csv`

{to_md(before_profile.rename_axis("column"))}

## Cleaned data (after)

File: `data/cleaned/sales_cleaned.csv`

{to_md(after_profile.rename_axis("column"))}

## Text values after cleaning

- **city**: {", ".join(sorted(clean["city"].unique()))}
- **product**: {", ".join(sorted(clean["product"].unique()))}
- **payment_method**: {", ".join(sorted(clean["payment_method"].unique()))}

## Number columns after cleaning

{to_md(clean[["quantity", "unit_price", "total_amount"]].describe().round(2).rename_axis("stat"))}

## What changed (cleaning log)

{to_md(log_table.rename_axis("#"))}
"""

with open(PROFILE_FILE, "w", encoding="utf-8") as f:
    f.write(report)
print("Profile written to", PROFILE_FILE)

Profile written to ../docs/data_profile.md


## 10. Confirm the raw file was not changed

In [26]:
raw_hash_end = file_hash(RAW_FILE)
print("Hash at start:", raw_hash_start)
print("Hash at end:  ", raw_hash_end)

assert raw_hash_start == raw_hash_end, "The raw file was changed!"
print("Raw file is unchanged.")

Hash at start: 3aed2ae03719f83755896a5a79ccbd339f9580ac15c812ee14c7f79509256a56
Hash at end:   3aed2ae03719f83755896a5a79ccbd339f9580ac15c812ee14c7f79509256a56
Raw file is unchanged.
